# 3. Trainable Qiskit QNN

This notebook builds a six-qubit parameterized circuit with Qiskit and evaluates it using the local `StatevectorEstimator`, which returns exact expectation values. `EstimatorQNN` and `TorchConnector` connect the circuit to PyTorch optimization. No quantum hardware is used.

To keep the first run manageable, QNN training uses a reproducible subset of the saved training split; validation and test evaluation still use the saved validation/test files. Increase the subset or epochs only after the end-to-end checks work.

In [ ]:
from pathlib import Path
import sys, time, json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
sys.path.insert(0, str(Path.cwd().resolve()))
from main import load_split, batch_loss, evaluate_predictions, save_json

torch.manual_seed(2026)
np.random.seed(2026)
device = "cuda" if torch.cuda.is_available() else "cpu"
train = load_split("data/train.npz")
validation = load_split("data/validation.npz")
test = load_split("data/test.npz")
print("Saved split sizes:", len(train["observed"]), len(validation["observed"]), len(test["observed"]))

## Build the circuit

Each input bit controls an RY encoding rotation. Every trainable layer applies RY and RZ rotations to all six qubits, followed by nearest-neighbour RZZ entanglers. The readout features are six local Z expectations plus three neighbouring ZZ expectations.

In [ ]:
N_QUBITS = 6
N_LAYERS = 2
input_params = ParameterVector("x", N_QUBITS)
weight_params = ParameterVector("theta", N_LAYERS * N_QUBITS * 2)

qc = QuantumCircuit(N_QUBITS)
for qubit in range(N_QUBITS):
    qc.ry(np.pi * input_params[qubit], qubit)

cursor = 0
for layer in range(N_LAYERS):
    for qubit in range(N_QUBITS):
        qc.ry(weight_params[cursor], qubit); cursor += 1
        qc.rz(weight_params[cursor], qubit); cursor += 1
    for qubit in range(N_QUBITS - 1):
        qc.rzz(0.5, qubit, qubit + 1)

def pauli_string(occupied_qubits, n_qubits=6):
    # Qiskit Pauli strings list the highest-index qubit on the left.
    chars = ["I"] * n_qubits
    for qubit, pauli in occupied_qubits:
        chars[n_qubits - 1 - qubit] = pauli
    return "".join(chars)

observables = [SparsePauliOp(pauli_string([(q, "Z")])) for q in range(N_QUBITS)]
observables += [
    SparsePauliOp(pauli_string([(q, "Z"), (q + 1, "Z")]))
    for q in range(N_QUBITS - 1)
]
print(qc.draw("text"))
print("Number of expectation-value features:", len(observables))

In [ ]:
qnn = EstimatorQNN(
    circuit=qc,
    estimator=StatevectorEstimator(),
    input_params=list(input_params),
    weight_params=list(weight_params),
    observables=observables,
    input_gradients=True,
)
initial_weights = np.random.uniform(-0.1, 0.1, len(weight_params))
quantum_layer = TorchConnector(qnn, initial_weights=initial_weights)

class HybridQNN(nn.Module):
    def __init__(self, quantum_layer):
        super().__init__()
        self.quantum_layer = quantum_layer
        self.head = nn.Linear(len(observables), 16)
    def forward(self, x):
        features = self.quantum_layer(x)
        return self.head(features).view(-1, 2, 8)

model = HybridQNN(quantum_layer).to(device)
probe_x = torch.tensor(train["observed"][:4, -1, :], dtype=torch.float32, device=device)
probe_y = torch.tensor(np.stack([train["labels_x"][:4], train["labels_z"][:4]], axis=1),
                       dtype=torch.long, device=device)
probe_out = model(probe_x)
probe_loss = batch_loss(probe_out, probe_y)
probe_loss.backward()
assert probe_out.shape == (4, 2, 8)
assert torch.isfinite(probe_loss)
assert model.quantum_layer.weight.grad is not None
assert torch.isfinite(model.quantum_layer.weight.grad).all()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
print("Output shape:", tuple(probe_out.shape), "Finite loss:", float(probe_loss))

## Train the hybrid QNN

The circuit's trainable angles and the classical output layer are optimized together with Adam. Exact expectation values avoid shot noise in this baseline. Quantum circuit gradients are calculated through Qiskit Machine Learning's estimator-QNN integration.

In [ ]:
def qnn_features(split):
    # The QNN is a single-round model, like the MLP.
    x = split["observed"][:, -1, :].astype(np.float32)
    y = np.stack([split["labels_x"], split["labels_z"]], axis=1).astype(np.int64)
    return x, y

x_train, y_train = qnn_features(train)
x_val, y_val = qnn_features(validation)
# A small fixed subset makes the exact simulator experiment more approachable.
subset_size = min(512, len(x_train))
x_train_small, y_train_small = x_train[:subset_size], y_train[:subset_size]
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train_small), torch.from_numpy(y_train_small)),
                          batch_size=16, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
initial_quantum_weights = model.quantum_layer.weight.detach().clone()
history = {"train_loss": [], "validation_loss": [], "validation_accuracy": []}
best_val = -1.0
started = time.perf_counter()

for epoch in range(8):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        before = model.quantum_layer.weight.detach().clone()
        optimizer.zero_grad()
        logits = model(xb)
        loss = batch_loss(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))
    model.eval()
    with torch.no_grad():
        val_logits = []
        for start in range(0, len(x_val), 32):
            xb = torch.tensor(x_val[start:start+32], dtype=torch.float32, device=device)
            val_logits.append(model(xb).cpu().numpy())
        val_logits = np.concatenate(val_logits)
        val_loss = float(batch_loss(torch.tensor(val_logits), torch.tensor(y_val)).item())
    metrics = evaluate_predictions(val_logits, y_val)
    history["train_loss"].append(float(np.mean(losses)))
    history["validation_loss"].append(val_loss)
    history["validation_accuracy"].append(metrics["correction_label_accuracy"])
    print(f"QNN epoch {epoch+1}/8: train_loss={np.mean(losses):.4f}, "
          f"val_loss={val_loss:.4f}, val_exact={metrics['correction_label_accuracy']:.3f}")
    if metrics["correction_label_accuracy"] > best_val:
        best_val = metrics["correction_label_accuracy"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "n_layers": N_LAYERS,
            "n_qubits": N_QUBITS,
            "history": history,
            "best_validation_accuracy": best_val,
            "input_params": [str(p) for p in input_params],
            "weight_params": [str(p) for p in weight_params],
        }, "models/qnn.pt")

history["training_seconds"] = time.perf_counter() - started

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["validation_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("QNN training history")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/qnn_loss.png", dpi=150)
plt.show()
print("Saved best checkpoint to models/qnn.pt")

## Summary and validation notes

The notebook checks output dimensions, finite loss, and finite gradients before training. It saves the QNN checkpoint and training history. The evaluation notebook reconstructs the same circuit and class, loads the checkpoint, and evaluates unseen test samples. This is a local exact-simulator result, not a hardware run.